In [1]:
# ==============================================================================
# MODULE 1.1: STOCHASTIC GPS PING SIMULATION ENGINE (OLTP PRODUCTION ENVIRONMENT)
# Objective: Generate high-frequency geospatial data (10s pings) for March 2026.
# Incorporates weather friction anomalies and dynamic urban traffic bottlenecks.
# ==============================================================================

import os
import math
import random
import pandas as pd
import numpy as np

# 1. Operational Parameters Configuration
NUM_DAYS = 31
NUM_DRIVERS = 5000
STEP_BASE = 0.00063  # Equivalent to ~70 meters displacement per 10-second interval

# 2. Geographic Boundary Conditions (Reforma-Centro Corridor, CDMX)
LAT_MIN, LAT_MAX = 19.4181864, 19.4366708
LNG_MIN, LNG_MAX = -99.1798285, -99.1439186

# 3. Macroeconomic Friction & Traffic Bottleneck Coordinates
LAT_ANGEL, LNG_ANGEL = 19.426953, -99.167621          # Angel of Independence Epicenter
LAT_CHAPULTEPEC, LNG_CHAPULTEPEC = 19.4222389, -99.1781452  # Chapultepec Underpass Epicenter
RADIO_BLOQUEO_GRADOS = 0.0018  # Spatial impact buffer (~200 meters radius)

# 4. Stochastic Environmental Variables (CONAGUA Precipitation Patterns)
# Random selection of exactly 6 precipitation days distributed across the month
DIAS_CON_LLUVIA = random.sample(range(1, NUM_DAYS + 1), 6)

# 5. Ingestion of the Padrón Base (Master Driver Registry)
drivers_catalog_path = "/kaggle/input/movilidad-cdmx-catalogos/drivers.csv"
if os.path.exists(drivers_catalog_path):
    df_drivers_origin = pd.read_csv(drivers_catalog_path)
    driver_ids = df_drivers_origin['id_driver'].tolist()
else:
    driver_ids = [f"DRV_{i:05d}" for i in range(1, NUM_DRIVERS + 1)]

# 6. Fleet Shift Scheduling Engine (Probability Distribution Matrix)
shift_categories = ['Off-Duty', 'Morning (06-14)', 'Afternoon (14-22)', 'Reinforcement (12-20)', 'Night (22-06)']
shift_probabilities = [0.10, 0.35, 0.30, 0.15, 0.10]

assignment_days, assignment_drivers, assignment_shifts = [], [], []
for day in range(1, NUM_DAYS + 1):
    daily_shifts = random.choices(shift_categories, weights=shift_probabilities, k=NUM_DRIVERS)
    for i in range(NUM_DRIVERS):
        assignment_days.append(day)
        assignment_drivers.append(driver_ids[i])
        assignment_shifts.append(daily_shifts[i])

df_shift_schedule = pd.DataFrame({
    'operational_day': assignment_days, 
    'id_driver': assignment_drivers, 
    'assigned_shift': assignment_shifts
})

# 7. Core Stochastic Simulation & Spatiotemporal Ingestion Engine
print(f"[SYSTEM] Initializing GPS Engine for March 2026...")
print(f"[METEOROLOGY] Stochastic rain events scheduled for days: {sorted(DIAS_CON_LLUVIA)}")

# Spatial memory state retention dictionary for active tracking
current_positions = {driver: {'lat': None, 'lng': None} for driver in driver_ids}

# Initialize data lake raw directory
os.makedirs("/kaggle/working/pings_raw", exist_ok=True)

for day in range(1, NUM_DAYS + 1):
    print(f" -> Processing Day {day:02d}/{NUM_DAYS}...", end="")
    
    df_today_shifts = df_shift_schedule[df_shift_schedule['operational_day'] == day]
    active_shifts_dict = dict(zip(df_today_shifts['id_driver'], df_today_shifts['assigned_shift']))
    
    # Exogenous Climatological Matrix Setup
    is_rainy_day = day in DIAS_CON_LLUVIA
    rain_start_hour = 17 if is_rainy_day else -1  # Afternoon commute precipitation window
    rain_duration_hours = 3 if is_rainy_day else 0
    
    # Exogenous Road Blockade Matrix Setup (30% Daily Probability)
    has_blockade_today = random.random() < 0.30
    blockade_location = random.choice(['ANGEL', 'CHAPULTEPEC']) if has_blockade_today else None
    blockade_start_hour = random.randint(8, 20) if has_blockade_today else -1
    blockade_duration_hours = random.randint(1, 3) if has_blockade_today else 0
    
    chunk_drivers, chunk_timestamps, chunk_latitudes, chunk_longitudes = [], [], [], []
    
    for hour in range(24):
        # Environmental Friction Evaluation (Moderate Rain reduces velocity bound by 30%)
        is_rain_active = is_rainy_day and (rain_start_hour <= hour < rain_start_hour + rain_duration_hours)
        weather_friction_factor = 0.7 if is_rain_active else 1.0
        
        # Fleet Availability State Machine Integration
        hourly_active_drivers = []
        for driver in driver_ids:
            current_shift = active_shifts_dict[driver]
            if current_shift == 'Morning (06-14)' and (6 <= hour < 14): hourly_active_drivers.append(driver)
            elif current_shift == 'Afternoon (14-22)' and (14 <= hour < 22): hourly_active_drivers.append(driver)
            elif current_shift == 'Reinforcement (12-20)' and (12 <= hour < 20): hourly_active_drivers.append(driver)
            elif current_shift == 'Night (22-06)' and (hour >= 22 or hour < 6): hourly_active_drivers.append(driver)
        
        # Dispatch Simulation Entry (Fleet deployment into geographic box)
        for driver in hourly_active_drivers:
            if current_positions[driver]['lat'] is None:
                current_positions[driver]['lat'] = random.uniform(LAT_MIN, LAT_MAX)
                current_positions[driver]['lng'] = random.uniform(LNG_MIN, LNG_MAX)
                
        # Deprovisioning State Engine (Fleet terminal off-duty state reset)
        for driver in driver_ids:
            if driver not in hourly_active_drivers:
                current_positions[driver]['lat'] = None
                current_positions[driver]['lng'] = None

        # Temporal Base Construction
        timestamp_base = pd.Timestamp(f'2026-03-{day:02d} {hour:02d}:00:00')
        
        # High-Frequency Sub-Interval Loop (360 steps per hour = 10-second updates)
        for step in range(360):
            step_timestamp = timestamp_base + pd.Timedelta(seconds=step * 10)
            is_blockade_active = has_blockade_today and (blockade_start_hour <= hour < blockade_start_hour + blockade_duration_hours)
            
            for driver in hourly_active_drivers:
                lat_current = current_positions[driver]['lat']
                lng_current = current_positions[driver]['lng']
                
                # Base Spatial Displacement Bound
                max_allowed_step = STEP_BASE * weather_friction_factor
                
                # Localized Structural Friction Evaluation (Congestion near accidents)
                if is_blockade_active:
                    if blockade_location == 'ANGEL':
                        distance_to_accident = math.sqrt((lat_current - LAT_ANGEL)**2 + (lng_current - LNG_ANGEL)**2)
                    else:
                        distance_to_accident = math.sqrt((lat_current - LAT_CHAPULTEPEC)**2 + (lng_current - LNG_CHAPULTEPEC)**2)
                        
                    if distance_to_accident <= RADIO_BLOQUEO_GRADOS:
                        max_allowed_step = max_allowed_step * 0.15  # Severe structural bottleneck (85% speed penalty)
                
                # Trigonometric Stochastic Walk Engine
                direction_theta = random.uniform(0, 2 * math.pi)
                step_magnitude = random.uniform(0, max_allowed_step)
                lat_new = lat_current + math.cos(direction_theta) * step_magnitude
                lng_new = lng_current + math.sin(direction_theta) * step_magnitude
                
                # Boundary Elastic Reflection Implementation
                if lat_new < LAT_MIN or lat_new > LAT_MAX: lat_new = lat_current - math.cos(direction_theta) * step_magnitude
                if lng_new < LNG_MIN or lng_new > LNG_MAX: lng_new = lng_current - math.sin(direction_theta) * step_magnitude
                
                current_positions[driver]['lat'] = lat_new
                current_positions[driver]['lng'] = lng_new
                
                chunk_drivers.append(driver)
                chunk_timestamps.append(step_timestamp)
                chunk_latitudes.append(lat_new)
                chunk_longitudes.append(lng_new)

    # Convert operational data slice into an optimal columnar schema
    df_daily_pings = pd.DataFrame({
        'id_driver': chunk_drivers, 
        'timestamp': chunk_timestamps, 
        'latitude': chunk_latitudes, 
        'longitude': chunk_longitudes
    })
    
    daily_output_path = f"/kaggle/working/pings_raw/pings_dia_{day:02d}.parquet"
    df_daily_pings.to_parquet(daily_output_path, compression='snappy', index=False)
    print("Snappy-Parquet File Committed Successfully.")

print("\n MARCH 2026 OPERATIONAL DATA GENERATION COMPLETED.")


[SYSTEM] Initializing GPS Engine for March 2026...
[METEOROLOGY] Stochastic rain events scheduled for days: [3, 19, 22, 25, 30, 31]
 -> Processing Day 01/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 02/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 03/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 04/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 05/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 06/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 07/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 08/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 09/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 10/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 11/31...Snappy-Parquet File Committed Successfully.
 -> Processing Day 12/31...Snappy-Parquet File Committed Successfully.
 -> Processing D

In [2]:
# ==============================================================================
# MODULE 1.2: USER DEMAND & REAL-TIME QUOTE ENGINE (OLTP PRODUCTION ENVIRONMENT)
# Objective: Simulate user search intents and price elasticity for March 2026.
# Captures latent demand behavioral profiles via elastic rejection state metrics.
# ==============================================================================

# 1. Operational Fleet & User Parameters
NUM_DAYS = 31
NUM_USERS = 100000
TOTAL_BASE_INTENTS = 3000000

# Seed synchronization for absolute data lake consistency
random.seed(42)
np.random.seed(42)

# Geographic Boundary Conditions (Reforma-Centro Corridor, CDMX)
LAT_MIN, LAT_MAX = 19.4181864, 19.4366708
LNG_MIN, LNG_MAX = -99.1798285, -99.1439186

# Climatological Timeline Alignment (6 Fixed Precipitation Days)
DIAS_CON_LLUVIA = [3, 7, 12, 18, 24, 29]

# 2. Ingestion of Master User Registry (Foreign Key Enforcement)
users_catalog_path = "/kaggle/input/movilidad-cdmx-catalogos/users.csv"
if os.path.exists(users_catalog_path):
    df_users_origin = pd.read_csv(users_catalog_path)
    user_ids = df_users_origin['id_user'].tolist()
else:
    user_ids = [f"USR_{i:06d}" for i in range(1, NUM_USERS + 1)]

# 3. Behavioral Cohort Segmentation (Psychological Profile Mapping)
shuffled_users = user_ids.copy()
random.shuffle(shuffled_users)

corporate_threshold = int(NUM_USERS * 0.30)
nocturnal_threshold = corporate_threshold + int(NUM_USERS * 0.20)

profile_corporate = set(shuffled_users[:corporate_threshold])
profile_nocturnal = set(shuffled_users[corporate_threshold:nocturnal_threshold])
profile_occasional = set(shuffled_users[nocturnal_threshold:])

# 4. Temporal Distribution & Traffic Weight Matrix Formulation
print("[SYSTEM] Building temporal simulation matrices for user quote intents...")

# Peak commuter weights allocated to corporate rush hours (08:00-10:00 and 18:00-20:00)
hourly_weights_raw = [
    0.01, 0.005, 0.005, 0.005, 0.01, 0.02,  # 00:00 - 05:00 (Low-Frequency Off-Peak)
    0.05, 0.15, 0.12, 0.06, 0.04, 0.04,   # 06:00 - 11:00 (Morning Rush Peak Window)
    0.05, 0.04, 0.04, 0.05, 0.05, 0.06,   # 12:00 - 17:00 (Mid-Day Standard Operations)
    0.14, 0.11, 0.06, 0.04, 0.02, 0.01    # 18:00 - 23:00 (Evening Evening Rush Peak Window)
]

# Normalization layer to guarantee statistical probability sums exactly to 1.0
hourly_probabilities = [weight / sum(hourly_weights_raw) for weight in hourly_weights_raw]

# Broadcast baseline intents across the operational calendar
sampled_days = np.random.randint(1, NUM_DAYS + 1, TOTAL_BASE_INTENTS)
sampled_hours = np.random.choice(range(24), size=TOTAL_BASE_INTENTS, p=hourly_probabilities)

# 5. Global Collection Ingestion Arrays
c_search, c_user, c_time, c_car, c_olat, c_olng, c_dlat, c_dlng, c_base, c_quoted, c_status = [[] for _ in range(11)]

# 6. Exogenous Daily Incident Timeline Generation
daily_blockade_schedule = {}
for day in range(1, NUM_DAYS + 1):
    has_blockade = random.random() < 0.30
    daily_blockade_schedule[day] = {
        'active': has_blockade,
        'start_hour': random.randint(8, 20) if has_blockade else -1,
        'duration_hours': random.randint(1, 3) if has_blockade else 0
    }

print("[SIMULATION] Executing price estimation and consumer behavior engine...")
for day in range(1, NUM_DAYS + 1):
    day_mask = (sampled_days == day)
    hourly_intents_today = sampled_hours[day_mask]
    total_daily_intents = len(hourly_intents_today)
    
    is_rainy_day = day in DIAS_CON_LLUVIA
    rain_start_hour = 17 if is_rainy_day else -1
    rain_duration_hours = 3 if is_rainy_day else 0
    
    current_blockade = daily_blockade_schedule[day]
    search_sequence_counter = len(c_search) + 1
    
    for idx in range(total_daily_intents):
        hour = hourly_intents_today[idx]
        selected_user = random.choice(user_ids)
        
        # Micro-Burst Intent Simulation based on User Psychometrics
        is_rush_hour = (8 <= hour < 10) or (18 <= hour < 20)
        if selected_user in profile_corporate and is_rush_hour:
            burst_attempts = random.randint(3, 5)  # Simulates high quote-refresh anxiety
        elif selected_user in profile_nocturnal and hour >= 22:
            burst_attempts = 1
        else:
            burst_attempts = random.randint(1, 2)
            
        # Geographic Route Node Assignments
        origin_lat = random.uniform(LAT_MIN, LAT_MAX)
        origin_lng = random.uniform(LNG_MIN, LNG_MAX)
        destination_lat = random.uniform(LAT_MIN, LAT_MAX)
        destination_lng = random.uniform(LNG_MIN, LNG_MAX)
        
        # Simplified Haversine Distance Estimation for Central Mexico City
        linear_distance_km = math.sqrt((origin_lat - destination_lat)**2 + (origin_lng - destination_lng)**2) * 111.1
        if linear_distance_km < 0.5: 
            linear_distance_km = 0.5  # Floor constraint for terminal operations
            
        # Vehicle Fleet Category Distribution Matrix
        service_tier = random.choices(['Standard', 'XL', 'Black'], weights=[0.75, 0.15, 0.10], k=1)[0]
        vehicle_multiplier = 1.0 if service_tier == 'Standard' else (1.5 if service_tier == 'XL' else 2.2)
        
        # Base Financial Pricing Model Equation
        distance_fee = linear_distance_km * 12.0
        duration_fee = (linear_distance_km * 2.5) * 3.50
        calculated_base_fare = (35.0 + distance_fee + duration_fee) * vehicle_multiplier
        
        base_seconds_seed = random.randint(0, 50)
        
        for attempt in range(burst_attempts):
            id_search_string = f"SCH_{search_sequence_counter:07d}"
            search_sequence_counter += 1
            
            # Incremental sequential timing for burst refresh sessions
            base_seconds_seed += random.randint(20, 40)
            minute_overflow = base_seconds_seed // 60
            final_seconds = base_seconds_seed % 60
            final_minutes = (random.randint(0, 54) + minute_overflow) % 60
            
            intent_timestamp = pd.Timestamp(f'2026-03-{day:02d} {hour:02d}:{final_minutes:02d}:{final_seconds:02d}')
            
            # Structural Surge Pricing Calculations
            is_rain_active = is_rainy_day and (rain_start_hour <= hour < rain_start_hour + rain_duration_hours)
            is_blockade_active = current_blockade['active'] and (current_blockade['start_hour'] <= hour < current_blockade['start_hour'] + current_blockade['duration_hours'])
            
            dynamic_surge_multiplier = 1.0
            if is_blockade_active: 
                dynamic_surge_multiplier += random.uniform(0.3, 0.6)
            if is_rain_active: 
                dynamic_surge_multiplier += random.uniform(0.4, 0.9)
            if dynamic_surge_multiplier > 2.2: 
                dynamic_surge_multiplier = 2.2  # Dynamic cap for the March regulatory model
                
            final_quoted_fare = calculated_base_fare * dynamic_surge_multiplier
            
            # Demand Elasticity & Consumer Conversion State Machine
            if dynamic_surge_multiplier == 1.0:
                conversion_status = 'solicited' if random.random() < 0.88 else 'abandoned'
            else:
                # Power-law decay modeling drop in conversion as surge increases
                conversion_probability = 0.88 / (dynamic_surge_multiplier ** 2.5)
                conversion_status = 'solicited' if random.random() < conversion_probability else 'abandoned'
                
            c_search.append(id_search_string)
            c_user.append(selected_user)
            c_time.append(intent_timestamp)
            c_car.append(service_tier)
            c_olat.append(round(origin_lat, 6))
            c_olng.append(round(origin_lng, 6))
            c_dlat.append(round(destination_lat, 6))
            c_dlng.append(round(destination_lng, 6))
            c_base.append(round(calculated_base_fare, 2))
            c_quoted.append(round(final_quoted_fare, 2))
            c_status.append(conversion_status)

# 7. Columnar Data Structure Construction
df_search_registry = pd.DataFrame({
    'id_search': c_search, 'id_user': c_user, 'timestamp': c_time, 'car_type_requested': c_car,
    'origin_latitude': c_olat, 'origin_longitude': c_olng, 'destination_latitude': c_dlat, 'destination_longitude': c_dlng,
    'price_base': c_base, 'price_quoted': c_quoted, 'status': c_status
})

# Chronological sorting for structural data lake ordering
df_search_registry = df_search_registry.sort_values('timestamp').reset_index(drop=True)

# Commit optimized slice to storage disk
os.makedirs("/kaggle/working/searches_raw", exist_ok=True)
parquet_output_path = "/kaggle/working/searches_raw/search_app_mes.parquet"
df_search_registry.to_parquet(parquet_output_path, compression='snappy', index=False)

print("\n" + "="*60)
print("DEMAND INTENT METRICS AUDIT (MARCH 2026)")
print("="*60)
print(df_search_registry['status'].value_counts())
print("-" * 60)
print(f"[SUCCESS] Table search_app committed. Total rows written: {len(df_search_registry):,}")


[SYSTEM] Building temporal simulation matrices for user quote intents...
[SIMULATION] Executing price estimation and consumer behavior engine...

DEMAND INTENT METRICS AUDIT (MARCH 2026)
status
solicited    4393889
abandoned     912632
Name: count, dtype: int64
------------------------------------------------------------
[SUCCESS] Table search_app committed. Total rows written: 5,306,521


In [3]:
# ==============================================================================
# MODULE 1.3: OPERATIONAL TRIP CONVERSION ENGINE (OLTP PRODUCTION ENVIRONMENT)
# Objective: Process conversion logic for user requested trips in March 2026.
# Enforces Third Normal Form (3FN) and captures fulfillment vs fulfillment failures.
# ==============================================================================

# 1. Operational Parameters Configuration
NUM_DAYS = 31
NUM_DRIVERS = 5000

# Seed synchronization for absolute data lake consistency
random.seed(42)
np.random.seed(42)

# Macroeconomic Friction & Traffic Bottleneck Coordinates
LAT_ANGEL, LNG_ANGEL = 19.427025, -99.167644
LAT_CHAPULTEPEC, LNG_CHAPULTEPEC = 19.4222389, -99.1781452
RADIO_BLOQUEO_GRADOS = 0.0018

# 2. Ingestion of the Validated User Search Registry
searches_input_path = "/kaggle/working/searches_raw/search_app_mes.parquet"
if os.path.exists(searches_input_path):
    df_searches = pd.read_parquet(searches_input_path)
else:
    df_searches = df_search_registry  # Fallback to current memory dataframe

# Extract strict subset of user conversions ('solicited' records only)
df_converted_trips = df_searches[df_searches['status'] == 'solicited'].copy()

# Sort chronologically to preserve transactional consistency
df_converted_trips = df_converted_trips.sort_values('timestamp').reset_index(drop=True)

print(f"[SYSTEM] Converted trip records found for processing: {len(df_converted_trips):,}")

# 3. Climatological & Incident Calendar Mapping (March 2026 Timeline Sync)
DIAS_CON_LLUVIA = [2, 4, 15, 23, 27, 28]

daily_incident_matrix = {}
for day in range(1, NUM_DAYS + 1):
    has_blockade = random.random() < 0.30
    daily_incident_matrix[day] = {
        'active': has_blockade,
        'start_hour': random.randint(8, 20) if has_blockade else -1,
        'duration_hours': random.randint(1, 3) if has_blockade else 0,
        'location': random.choice(['ANGEL', 'CHAPULTEPEC']) if has_blockade else None
    }

# 4. Global Collection Ingestion Arrays
c_trip, c_search, c_driver, c_start, c_end, c_status = [[] for _ in range(6)]

print("[MATCHING] Initializing fleet fulfillment assignment and delay engine...")
driver_fleet_pool = [f"DRV_{i:05d}" for i in range(1, NUM_DRIVERS + 1)]

# 5. Core Transactional Loop: Processing Converted Records
for idx, row in df_converted_trips.iterrows():
    id_trip_string = f"TRP_{idx+1:07d}"
    id_search_fk = row['id_search']
    timestamp_start = row['timestamp']
    
    current_day = timestamp_start.day
    current_hour = timestamp_start.hour
    
    origin_lat, origin_lng = row['origin_latitude'], row['origin_longitude']
    destination_lat, destination_lng = row['destination_latitude'], row['destination_longitude']
    
    # Baseline Driver Acceptance Rate Constraint (75% Fulfillment Target)
    # 25% of requests result in unassigned states due to local inventory deficits
    if random.random() > 0.75:
        c_trip.append(id_trip_string)
        c_search.append(id_search_fk)
        c_driver.append(None)       # Stored as database NULL state
        c_start.append(timestamp_start)
        c_end.append(None)          # Stored as database NULL state
        c_status.append('unassigned')
        continue
    
    # Fleet Assignment Optimization via Random Pool Sampling
    assigned_driver = random.choice(driver_fleet_pool)
    
    # Physical Velocity & Congestion Degradation Modeling
    trip_distance_km = math.sqrt((origin_lat - destination_lat)**2 + (origin_lng - destination_lng)**2) * 111.1
    current_velocity_km_min = 25.0 / 60.0  # Standard urban transit benchmark: 25 km/h
    
    is_rain_active = current_day in DIAS_CON_LLUVIA and (17 <= current_hour < 20)
    
    active_incident = daily_incident_matrix[current_day]
    is_blockade_active = active_incident['active'] and (active_incident['start_hour'] <= current_hour < active_incident['start_hour'] + active_incident['duration_hours'])
    
    crosses_structural_bottleneck = False
    if is_blockade_active:
        target_lat = LAT_ANGEL if active_incident['location'] == 'ANGEL' else LAT_CHAPULTEPEC
        target_lng = LNG_ANGEL if active_incident['location'] == 'ANGEL' else LNG_CHAPULTEPEC
        distance_to_blockade = math.sqrt((origin_lat - target_lat)**2 + (origin_lng - target_lng)**2)
        if distance_to_blockade <= RADIO_BLOQUEO_GRADOS:
            crosses_structural_bottleneck = True
            
    if is_rain_active: 
        current_velocity_km_min = current_velocity_km_min * 0.70  # 30% environmental penalty
    if crosses_structural_bottleneck: 
        current_velocity_km_min = current_velocity_km_min * 0.15  # 85% localized bottleneck gridlock penalty
        
    calculated_trip_minutes = trip_distance_km / current_velocity_km_min
    if calculated_trip_minutes < 3.0: 
        calculated_trip_minutes = 3.0  # Terminal buffer for traffic signaling constraints
    
    # Passenger & Driver Friction State Machine (Time-Out Cancellation Modeling)
    if calculated_trip_minutes > 35.0 and random.random() < 0.15:
        operational_status = 'canceled'
        timestamp_end = timestamp_start + pd.Timedelta(minutes=4)  # Cancellation penalty constraint
    else:
        operational_status = 'completed'
        timestamp_end = timestamp_start + pd.Timedelta(minutes=calculated_trip_minutes)
        
    c_trip.append(id_trip_string)
    c_search.append(id_search_fk)
    c_driver.append(assigned_driver)
    c_start.append(timestamp_start)
    c_end.append(timestamp_end)
    c_status.append(operational_status)

# 6. Columnar Structure Conversion
df_trip_registry = pd.DataFrame({
    'id_trip': c_trip, 
    'id_search': c_search, 
    'id_driver': c_driver,
    'time_start': c_start, 
    'time_end': c_end, 
    'status': c_status
})

# Export transactional state to disk
os.makedirs("/kaggle/working/trips_raw", exist_ok=True)
parquet_output_path = "/kaggle/working/trips_raw/trips_mes.parquet"
df_trip_registry.to_parquet(parquet_output_path, compression='snappy', index=False)

print("\n" + "="*60)
print("FLEET OPERATIONS FULFILLMENT AUDIT (MARCH 2026)")
print("="*60)
print(df_trip_registry['status'].value_counts())
print("-" * 60)
print(f"Table trips committed in 3FN. Total rows written: {len(df_trip_registry):,}")


[SYSTEM] Converted trip records found for processing: 4,393,889
[MATCHING] Initializing fleet fulfillment assignment and delay engine...

FLEET OPERATIONS FULFILLMENT AUDIT (MARCH 2026)
status
completed     3295534
unassigned    1098301
canceled           54
Name: count, dtype: int64
------------------------------------------------------------
Table trips committed in 3FN. Total rows written: 4,393,889
